In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import cv2
import numpy as np

!pip install -qq git+https://github.com/udesa-vision/i308-utils.git

from i308_utils import imshow, show_images
import matplotlib.pyplot as plt

In [ ]:
# Configs

FACTOR = 0.25
N_ANMS = 400
RATIO = 0.8
T_RANSAC = 3
N_RANSAC = 2000

### 3.1

In [ ]:
img_izq = cv2.imread('inputs/cuadro_0.jpg', cv2.IMREAD_GRAYSCALE)
img_centro = cv2.imread('inputs/cuadro_1.jpg', cv2.IMREAD_GRAYSCALE)
img_der = cv2.imread('inputs/cuadro_2.jpg', cv2.IMREAD_GRAYSCALE)

img_izq = cv2.resize(img_izq, None, fx=FACTOR, fy=FACTOR, interpolation=cv2.INTER_AREA)
img_centro = cv2.resize(img_centro, None, fx=FACTOR, fy=FACTOR, interpolation=cv2.INTER_AREA)
img_der = cv2.resize(img_der, None, fx=FACTOR, fy=FACTOR, interpolation=cv2.INTER_AREA)

kp_izq, des_izq = cv2.SIFT_create().detectAndCompute(img_izq, None)
kp_centro, des_centro = cv2.SIFT_create().detectAndCompute(img_centro, None)
kp_der, des_der = cv2.SIFT_create().detectAndCompute(img_der, None)

In [ ]:
kp_image_izq = cv2.cvtColor(img_izq, cv2.COLOR_GRAY2BGR)
kp_image_centro = cv2.cvtColor(img_centro, cv2.COLOR_GRAY2BGR)
kp_image_der = cv2.cvtColor(img_der, cv2.COLOR_GRAY2BGR)

# IZQ

kp_image_izq = cv2.drawKeypoints(
    kp_image_izq,
    kp_izq,
    0, (255, 0, 0),
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

# CENTRO

kp_image_centro = cv2.drawKeypoints(
    kp_image_centro,
    kp_centro,
    0, (255, 0, 0),
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

# DER

kp_image_der = cv2.drawKeypoints(
    kp_image_der,
    kp_der,
    0, (255, 0, 0),
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

show_images([
    kp_image_izq, kp_image_centro, kp_image_der
], ["keypoints izquierda", "keypoints centro", "keypoints derecha"], figsize=(18, 8))

### 3.2

In [ ]:
def anms(keypoints, N):
    Kl = len(keypoints)
    SD = 0
    R = np.full(Kl, np.inf)
    
    for i in range(Kl):
        for j in range(Kl):
            if keypoints[j].response > keypoints[i].response:
                xi, yi = keypoints[i].pt
                xj, yj = keypoints[j].pt
                SD = (xi - xj)**2 + (yi - yj)**2
                
                if SD < R[i]:
                    R[i] = SD
    
    best_R_indexes = np.argsort(R)[::-1]
    best_N_indexes = best_R_indexes[:N]
    
    return best_N_indexes

In [ ]:
indexes_izq = anms(kp_izq, N_ANMS)
indexes_centro = anms(kp_centro, N_ANMS)
indexes_der = anms(kp_der, N_ANMS)

In [ ]:
imagenes = [img_izq, img_centro, img_der]
keypoints = [kp_izq, kp_centro, kp_der]
indices = [indexes_izq, indexes_centro, indexes_der]
nombres = ['izquierda', 'centro', 'derecha']

kp_images_anms = []
for img, kps, idx in zip(imagenes, keypoints, indices):
    kp_image = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    kp_image = cv2.drawKeypoints(
        kp_image,
        [kps[i] for i in idx],
        0, (52, 104, 235),
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
    )
    kp_images_anms.append(kp_image)

show_images(
    kp_images_anms,
    [f"A-NMS {nombre}: {len(idx)} keypoints" for nombre, idx in zip(nombres, indices)],
    figsize=(18, 8)
)

In [ ]:
kp_izq_anms = [kp_izq[i] for i in indexes_izq]
kp_centro_anms = [kp_centro[i] for i in indexes_centro]
kp_der_anms = [kp_der[i] for i in indexes_der]

des_izq_anms = des_izq[indexes_izq]
des_centro_anms = des_centro[indexes_centro]
des_der_anms = des_der[indexes_der]

### 3.3

In [ ]:
def match_bf(des_query, des_train, crossCheck):
    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=crossCheck)
    return bf.match(des_query, des_train)


def plot_matches(matches_izq, matches_der, politica):
    res_img_izq = cv2.drawMatches(
        img_centro,
        kp_centro_anms,
        img_izq,
        kp_izq_anms,
        matches_izq,
        None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )

    res_img_der = cv2.drawMatches(
        img_centro,
        kp_centro_anms,
        img_der,
        kp_der_anms,
        matches_der,
        None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )

    show_images(
        [res_img_izq, res_img_der],
        [f"{politica} · centro <-> izquierda: {len(matches_izq)} matches",
         f"{politica} · centro <-> derecha: {len(matches_der)} matches"],
        grid=(2, 1), figsize=(14, 20)
    )

Fuerza Bruta (con y sin cross check)

In [ ]:
matches_izq_sin_filtro = match_bf(des_centro_anms, des_izq_anms, crossCheck=False)
matches_der_sin_filtro = match_bf(des_centro_anms, des_der_anms, crossCheck=False)

plot_matches(matches_izq_sin_filtro, matches_der_sin_filtro, "sin filtrar")

In [ ]:
matches_izq_cc = match_bf(des_centro_anms, des_izq_anms, crossCheck=True)
matches_der_cc = match_bf(des_centro_anms, des_der_anms, crossCheck=True)

plot_matches(matches_izq_cc, matches_der_cc, "cross-check")

In [ ]:
def match_lowe(des_query, des_train, ratio):
    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
    matches = bf.knnMatch(des_query, des_train, k=2)

    # Nos quedamos con matches cuyo segundo mejor match está lejos
    good_matches = []
    for m, n in matches:
        if m.distance < ratio * n.distance:
            good_matches.append(m)

    return good_matches

Lowe

In [ ]:
RATIOS = [0.5, 0.6, 0.7, 0.8]

matches_lowe = {}
for ratio in RATIOS:
    matches_izq_lowe = match_lowe(des_centro_anms, des_izq_anms, ratio)
    matches_der_lowe = match_lowe(des_centro_anms, des_der_anms, ratio)
    matches_lowe[ratio] = (matches_izq_lowe, matches_der_lowe)

    plot_matches(matches_izq_lowe, matches_der_lowe, f"Lowe {ratio}")

for ratio, (m_izq, m_der) in matches_lowe.items():
    print(f"ratio {ratio}: izquierda {len(m_izq)} · derecha {len(m_der)}")

Lowe + CrossCheck

In [ ]:
lowe_izq, lowe_der = matches_lowe[RATIO]
bfmatch_izq, bfmatch_der = matches_izq_cc, matches_der_cc

def interseccion(matches_lowe, matches_cc):
    pares_cc = {(m.queryIdx, m.trainIdx) for m in matches_cc}
    return [m for m in matches_lowe if (m.queryIdx, m.trainIdx) in pares_cc]

matches_izq_lowe_cc = interseccion(lowe_izq, bfmatch_izq)
matches_der_lowe_cc = interseccion(lowe_der, bfmatch_der)

plot_matches(matches_izq_lowe_cc, matches_der_lowe_cc, f"Lowe {ratio} + CC")

### 3.4

Der -> IDX: 1, 5, 6, 9
Izq -> IDX: 3, 4, 8, 13
Elegidos a mano haciendo plot_matches([matches_izq_lowe_cc[i]], [matches_der_lowe_cc[j]], f"Lowe {ratio} + CC")

Buscando que se cubra mucha area entre los puntos seleccionados, y no tener conjuntos de tres puntos que esten

In [ ]:
chosen_indexes_izq = [3, 4, 8, 13]
chosen_indexes_der = [1, 5, 6, 9]

chosen_matches_izq = [matches_izq_lowe_cc[i] for i in chosen_indexes_izq]
chosen_matches_der = [matches_der_lowe_cc[i] for i in chosen_indexes_der]

plot_matches(chosen_matches_izq, chosen_matches_der, f"4 Matches elegidos")

In [ ]:
def dlt(origen, destino):
    A = []
    b = []
    for i in range(4):
        x, y = origen[i]
        x_prima, y_prima = destino[i]
        A.append([-x, -y, -1, 0, 0, 0, x * x_prima, y * x_prima])
        A.append([0, 0, 0, -x, -y, -1, x * y_prima, y * y_prima])
        b.append(x_prima)
        b.append(y_prima)

    A = np.array(A)
    b = np.array(b)

    H = -np.linalg.solve(A, b)

    H = np.hstack([H, [1]])

    H = H.reshape(3, 3)

    return H

In [ ]:
origen_izq = np.float32([kp_izq_anms[m.trainIdx].pt for m in chosen_matches_izq])
destino_izq = np.float32([kp_centro_anms[m.queryIdx].pt for m in chosen_matches_izq])

origen_der = np.float32([kp_der_anms[m.trainIdx].pt for m in chosen_matches_der])
destino_der = np.float32([kp_centro_anms[m.queryIdx].pt for m in chosen_matches_der])

H_izq = dlt(origen_izq, destino_izq)
H_der = dlt(origen_der, destino_der)

### 3.5

In [ ]:
origen_izq_todos = np.float32([kp_izq_anms[m.trainIdx].pt for m in matches_izq_lowe_cc])
destino_izq_todos = np.float32([kp_centro_anms[m.queryIdx].pt for m in matches_izq_lowe_cc])

origen_der_todos = np.float32([kp_der_anms[m.trainIdx].pt for m in matches_der_lowe_cc])
destino_der_todos = np.float32([kp_centro_anms[m.queryIdx].pt for m in matches_der_lowe_cc])

In [ ]:
def homo(points_2d):
  n = points_2d.shape[0]
  z = np.ones((n, 1))
  return np.hstack((points_2d, z))

def cart(h_points):
  x = h_points[:, 0]
  y = h_points[:, 1]
  z = h_points[:, 2]
  return np.stack((x/z, y/z)).T

def apply_transform(shape, T):
  h_ori = homo(shape)
  h_dst = np.dot(T, h_ori.T)
  dst = cart(h_dst.T)
  return dst

def estimar_homografia_svd(ori, dst):
  A = []
  
  for i in range(len(ori)):
    x, y = ori[i]
    x_prima, y_prima = dst[i]
    A.append([-x, -y, -1, 0, 0, 0, x * x_prima, y * x_prima, x_prima])
    A.append([0, 0, 0, -x, -y, -1, x * y_prima, y * y_prima, y_prima])

  A = np.array(A)
  
  _, _, Vt = np.linalg.svd(A)
  
  h = Vt[-1, :]
  
  H = h.reshape(3, 3)
  
  return H

def RANSAC(origins, destinations, t, N):
  rng = np.random.default_rng(0)
  best_inliers = np.zeros(len(origins), dtype=bool)
  best_count = 0

  for _ in range(N):
    indexes_sample = rng.choice(len(origins), size=4, replace=False)
    origen_muestra = origins[indexes_sample]
    destino_muestra = destinations[indexes_sample]

    # Caso que se tomen datos repetidos y esto rompa
    try:
      H = dlt(origen_muestra, destino_muestra)
    except np.linalg.LinAlgError:
      continue

    errors = np.linalg.norm(apply_transform(origins, H) - destinations, axis=1)
    inliers_count = (errors < t).sum()

    if inliers_count > best_count:
      best_count = inliers_count
      best_inliers = errors < t
  
  best_origins = origins[best_inliers]
  best_destinations = destinations[best_inliers]
  
  H_final = estimar_homografia_svd(best_origins, best_destinations)
  
  # Opcional
  errors = np.linalg.norm(apply_transform(origins, H_final) - destinations, axis=1)
  best_inliers = errors < t
  
  best_origins = origins[best_inliers]
  best_destinations = destinations[best_inliers]
  
  H_final = estimar_homografia_svd(best_origins, best_destinations)
  H_final = H_final / H_final[2,2]
  
  return H_final, best_inliers

In [ ]:
H_izq_ransac, inliers_izq = RANSAC(origen_izq_todos, destino_izq_todos, T_RANSAC, N_RANSAC)
H_der_ransac, inliers_der = RANSAC(origen_der_todos, destino_der_todos, T_RANSAC, N_RANSAC)

izq_errs = np.linalg.norm(apply_transform(origen_izq_todos, H_izq_ransac) - destino_izq_todos, axis=1)
der_errs = np.linalg.norm(apply_transform(origen_der_todos, H_der_ransac) - destino_der_todos, axis=1)

izq_errs_final = izq_errs[inliers_izq]
der_errs_final = der_errs[inliers_der]

rmse_izq = np.sqrt(np.mean(izq_errs_final ** 2))
rmse_der = np.sqrt(np.mean(der_errs_final ** 2))

print(f"RMSE Izq <-> Centro {rmse_izq}px")
print(f"RMSE Der <-> Centro {rmse_der}px")

In [ ]:
def draw_inliers(matches, mask, img_lateral, kp_lateral):
    inliers = [m for m, es_inlier in zip(matches, mask) if es_inlier]
    outliers = [m for m, es_inlier in zip(matches, mask) if not es_inlier]
    flags = cv2.DrawMatchesFlags_DRAW_OVER_OUTIMG | cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS

    # imagen base con las dos fotos lado a lado, sin matches
    res = cv2.drawMatches(img_centro, kp_centro_anms, img_lateral, kp_lateral, [], None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    # primero outliers en rojo, después inliers en verde encima
    res = cv2.drawMatches(img_centro, kp_centro_anms, img_lateral, kp_lateral, outliers, res, matchColor=(0, 0, 255), flags=flags)
    res = cv2.drawMatches(img_centro, kp_centro_anms, img_lateral, kp_lateral, inliers, res, matchColor=(0, 255, 0), flags=flags)
    return res, len(inliers), len(outliers)

res_izq, n_in_izq, n_out_izq = draw_inliers(matches_izq_lowe_cc, inliers_izq, img_izq, kp_izq_anms)
res_der, n_in_der, n_out_der = draw_inliers(matches_der_lowe_cc, inliers_der, img_der, kp_der_anms)

show_images(
    [res_izq, res_der],
    [f"RANSAC · centro <-> izquierda: {n_in_izq} inliers (verde) · {n_out_izq} outliers (rojo)",
     f"RANSAC · centro <-> derecha: {n_in_der} inliers (verde) · {n_out_der} outliers (rojo)"],
    grid=(2, 1), figsize=(14, 20)
)

### 3.6

In [ ]:
# Proyectamos esquinas
h_izq, w_izq = img_izq.shape
h_der, w_der = img_der.shape
h_centro, w_centro = img_centro.shape

corners_izq = np.array([[0,0], [w_izq, 0], [w_izq, h_izq], [0, h_izq]])
corners_der = np.array([[0,0], [w_der, 0], [w_der, h_der], [0, h_der]])
corners_centro = np.array([[0,0], [w_centro, 0], [w_centro, h_centro], [0, h_centro]])

corners_izq_new = apply_transform(corners_izq, H_izq_ransac)
corners_der_new = apply_transform(corners_der, H_der_ransac)

In [ ]:
all_coords = np.vstack([corners_izq_new, corners_centro, corners_der_new])

min_x = np.floor(np.min(all_coords[:, 0]))
max_x = np.ceil(np.max(all_coords[:, 0]))

min_y = np.floor(np.min(all_coords[:, 1]))
max_y = np.ceil(np.max(all_coords[:, 1]))

w_whole = np.int32(max_x - min_x)
h_whole = np.int32(max_y - min_y)

print(f"Tamaño de la imagen completa: {w_whole} x {h_whole}")

T = np.array([[1, 0, -min_x], [0, 1, -min_y], [0, 0, 1]])

H_izq_ransac_new = T @ H_izq_ransac
H_der_ransac_new = T @ H_der_ransac
H_centro = T

traslated_corners_izq = apply_transform(corners_izq, H_izq_ransac_new)
traslated_corners_der = apply_transform(corners_der, H_der_ransac_new)
traslated_corners_centro = apply_transform(corners_centro, H_centro)

all_traslated_coords = np.vstack([traslated_corners_izq, traslated_corners_der, traslated_corners_centro])

print(f"Minimo y maximo de X con la traslacion aplicada: {np.floor(np.min(all_traslated_coords[:, 0]))}, {np.ceil(np.max(all_traslated_coords[:, 0]))}")
print(f"Minimo y maximo de Y con la traslacion aplicada: {np.floor(np.min(all_traslated_coords[:, 1]))}, {np.ceil(np.max(all_traslated_coords[:, 1]))}")

# Los maximos y minimos coinciden con el tamaño de la escena, trasladamos bien el centro :D

### 3.7

In [ ]:
img_izq_color = cv2.imread('inputs/cuadro_0.jpg', cv2.IMREAD_COLOR)
img_centro_color = cv2.imread('inputs/cuadro_1.jpg', cv2.IMREAD_COLOR)
img_der_color = cv2.imread('inputs/cuadro_2.jpg', cv2.IMREAD_COLOR)

img_izq_color = cv2.resize(img_izq_color, None, fx=FACTOR, fy=FACTOR, interpolation=cv2.INTER_AREA)
img_centro_color = cv2.resize(img_centro_color, None, fx=FACTOR, fy=FACTOR, interpolation=cv2.INTER_AREA)
img_der_color = cv2.resize(img_der_color, None, fx=FACTOR, fy=FACTOR, interpolation=cv2.INTER_AREA)

show_images([img_izq_color, img_centro_color, img_der_color], ["Izquierda", "Centro", "Derecha"])

In [ ]:
img_izq_warped = cv2.warpPerspective(img_izq_color, H_izq_ransac_new, (w_whole, h_whole))
img_centro_warped = cv2.warpPerspective(img_centro_color, H_centro, (w_whole, h_whole))
img_der_warped = cv2.warpPerspective(img_der_color, H_der_ransac_new, (w_whole, h_whole))

show_images([img_izq_warped, img_centro_warped, img_der_warped], ["Izquierda", "Centro", "Derecha"])

In [ ]:
img_izq_dims = img_izq_color.shape[:2]
img_centro_dims = img_centro_color.shape[:2]
img_der_dims = img_der_color.shape[:2]

img_izq_mask = np.zeros(img_izq_dims, dtype=np.uint8)
img_centro_mask = np.zeros(img_centro_dims, dtype=np.uint8)
img_der_mask = np.zeros(img_der_dims, dtype=np.uint8)

img_izq_mask[1:-1, 1:-1] = 255
img_centro_mask[1:-1, 1:-1] = 255
img_der_mask[1:-1, 1:-1] = 255

img_izq_mask = cv2.distanceTransform(img_izq_mask, cv2.DIST_L2, 5)
img_centro_mask = cv2.distanceTransform(img_centro_mask, cv2.DIST_L2, 5)
img_der_mask = cv2.distanceTransform(img_der_mask, cv2.DIST_L2, 5)

img_izq_mask_warped = cv2.warpPerspective(img_izq_mask, H_izq_ransac_new, (w_whole, h_whole))
img_centro_mask_warped = cv2.warpPerspective(img_centro_mask, H_centro, (w_whole, h_whole))
img_der_mask_warped = cv2.warpPerspective(img_der_mask, H_der_ransac_new, (w_whole, h_whole))

In [ ]:
imagenes_warped = np.stack([img_izq_warped, img_centro_warped, img_der_warped])
mascaras_warped = np.stack([img_izq_mask_warped, img_centro_mask_warped, img_der_mask_warped])[..., None] # Esto solo un atajo por prolijidad

numerador = (mascaras_warped * imagenes_warped).sum(axis=0)
denominador = mascaras_warped.sum(axis=0)

I = numerador / np.maximum(denominador, 1e-6)
I = np.clip(I, 0, 255).astype(np.uint8)

show_images([I], ["Panorámica"])

### DEMO

In [ ]:
from panorama import generar_panoramica

demo_img_izq = cv2.imread('inputs/udesa_0.jpg', cv2.IMREAD_COLOR)
demo_img_centro = cv2.imread('inputs/udesa_1.jpg', cv2.IMREAD_COLOR)
demo_img_der = cv2.imread('inputs/udesa_2.jpg', cv2.IMREAD_COLOR)

demo_pano = generar_panoramica(demo_img_izq, demo_img_centro, demo_img_der)

show_images([demo_pano], ["UdeSA"])

In [ ]:
demo_img_izq = cv2.imread('inputs/test_0.jpg', cv2.IMREAD_COLOR)
demo_img_centro = cv2.imread('inputs/test_1.jpg', cv2.IMREAD_COLOR)
demo_img_der = cv2.imread('inputs/test_2.jpg', cv2.IMREAD_COLOR)

demo_pano = generar_panoramica(demo_img_izq, demo_img_centro, demo_img_der)

show_images([demo_img_izq, demo_img_centro, demo_img_der], ["Izq", "Centro", "Der"])
show_images([demo_pano], ["Lavadero"])

### Números para el informe

Las celdas de abajo generan los valores que citamos en el informe y que no
quedan impresos en ninguna otra parte del notebook.

In [ ]:
# Sección 3.1: keypoints a resolución completa vs reducida
for nombre, archivo in [("izquierda", "cuadro_0"), ("centro", "cuadro_1"), ("derecha", "cuadro_2")]:
    img_full = cv2.imread(f'inputs/{archivo}.jpg', cv2.IMREAD_GRAYSCALE)
    kp_full, _ = cv2.SIFT_create().detectAndCompute(img_full, None)

    img_chico = cv2.resize(img_full, None, fx=FACTOR, fy=FACTOR, interpolation=cv2.INTER_AREA)
    kp_chico, _ = cv2.SIFT_create().detectAndCompute(img_chico, None)

    sizes = np.array([k.size for k in kp_full])
    print(f"{nombre}: {len(kp_full)} keypoints a resolución completa "
          f"({100 * (sizes < 4).mean():.0f}% de menos de 4 px) · {len(kp_chico)} al {100 * FACTOR:.0f}%")

In [ ]:
# Sección 4.2: homografía manual (3.4) contra RANSAC (3.5)
def consenso(H, origenes, destinos, t=T_RANSAC):
    errores = np.linalg.norm(apply_transform(origenes, H) - destinos, axis=1)
    return (errores < t).sum()


def rmse(H, origenes, destinos):
    errores = np.linalg.norm(apply_transform(origenes, H) - destinos, axis=1)
    return np.sqrt(np.mean(errores ** 2))


for nombre, H_manual, H_auto, O, D, inliers in [
    ("izquierda", H_izq, H_izq_ransac, origen_izq_todos, destino_izq_todos, inliers_izq),
    ("derecha", H_der, H_der_ransac, origen_der_todos, destino_der_todos, inliers_der),
]:
    for metodo, H in [("manual", H_manual), ("RANSAC", H_auto)]:
        print(f"{nombre} {metodo}: consenso {consenso(H, O, D)}/{len(O)} · "
              f"RMSE sobre inliers de RANSAC {rmse(H, O[inliers], D[inliers]):.2f} px")

In [ ]:
# Sección 4.3: los otros dos conjuntos con los mismos parámetros
for nombre, prefijo in [("UdeSA", "udesa"), ("propio", "test")]:
    imgs = [cv2.imread(f'inputs/{prefijo}_{i}.jpg', cv2.IMREAD_COLOR) for i in range(3)]
    pano, info = generar_panoramica(*imgs, info=True)

    datos = {clave: valor for clave, valor in info.items() if not str(clave).startswith("H_")}
    print(f"{nombre}: resolución original {imgs[0].shape[1]} x {imgs[0].shape[0]}")
    print("  ", datos)

    show_images([pano], [f"Conjunto {nombre}"])